# Survival MDN: train and inspect survival curves

This notebook demonstrates how to train `SurvivalMDN` on synthetic right-censored data, plot survival curves, inspect mixture parameters, and save and reload the model.

## Setup

Use Python 3.12 or 3.13. From the PyHealth repository root, install the project in editable mode:

```bash
python -m pip install -e .
python -m pip install jupyterlab ipykernel matplotlib pandas
python -m ipykernel install --user --name pyhealth-survival --display-name "PyHealth Survival"
python -m jupyterlab
```

Select the **PyHealth Survival** kernel. Open the notebook from this checkout; the setup below finds the repository from the current directory.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import pyhealth
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import SurvivalMDN
from pyhealth.trainer import Trainer

candidates = [
    p for base in (Path.cwd(), *Path.cwd().parents) for p in (base, base / "PyHealth")
]
repo = next(
    (p for p in candidates if (p / "examples/survival_mdn_synthetic.ipynb").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Open this notebook from the PyHealth checkout.")
if not Path(pyhealth.__file__).resolve().is_relative_to(repo.resolve()):
    raise RuntimeError(
        "Select a kernel with this checkout installed using pip install -e ."
    )

SEED = 0
EPOCHS = 10
DEVICE = "cpu"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| device:", DEVICE)
print("Repository:", repo)

## Synthetic data generator

Features determine a latent Gaussian mixture. Independent censoring produces the observed durations and event indicators. All times use arbitrary synthetic units.

In [ ]:
def _make_samples(seed, count=320, width=4):
    """Generate fixed-width features and independently censored event times."""
    rng = np.random.default_rng(seed)
    features = rng.normal(size=(count, width))
    probability = 1.0 / (1.0 + np.exp(-features[:, 0]))
    component = rng.random(count) < probability
    means = np.where(component, 1.2, -0.8) + 0.3 * features[:, 1]
    latent = means + 0.5 * rng.normal(size=count)
    event_times = np.logaddexp(0.0, latent)
    # Uniform draws strictly inside (0, 1) make censoring times positive.
    uniform = np.clip(rng.random(count), np.finfo(float).eps, 1.0 - 1e-16)
    censoring_times = -2.5 * np.log(uniform)
    durations = np.minimum(event_times, censoring_times)
    events = event_times <= censoring_times
    samples = [
        {
            "patient_id": f"synthetic-{index}",
            "features": features[index].tolist(),
            "duration": float(durations[index]),
            "event": float(events[index]),
        }
        for index in range(count)
    ]
    # The collator pads ragged vectors: validate source widths before processing.
    if any(len(sample["features"]) != width for sample in samples):
        raise ValueError("Every raw feature vector must have the same width")
    return samples

## 1. Inspect the observations

Each patient has four synthetic numeric features. The generator draws an event time from a transformed Gaussian mixture, then independently draws a censoring time. The observed `duration` is the smaller of those times.

- `event = 1`: the event occurred at the observed duration.
- `event = 0`: follow-up ended before the event; the later outcome is unknown.

All times use arbitrary synthetic units. Censored observations are useful training data, not negative examples of an event that will never occur.


In [ ]:
samples = _make_samples(SEED)
summary = pd.DataFrame(
    {
        "patient_id": [s["patient_id"] for s in samples],
        "duration": [s["duration"] for s in samples],
        "event": [int(s["event"]) for s in samples],
    }
)
display(summary.head(8))
print("Feature vector:", samples[0]["features"])
print(f"Patients: {len(samples)}; censored: {1 - summary.event.mean():.1%}")
fig, ax = plt.subplots(figsize=(7, 3))
for event, label in [(1, "Observed event"), (0, "Censored")]:
    ax.hist(
        summary.loc[summary.event == event, "duration"],
        bins=np.linspace(0, summary.duration.max(), 18),
        alpha=0.65,
        label=label,
    )
ax.set(
    xlabel="Observed duration (synthetic units)",
    ylabel="Patients",
    title="Observed events and censored follow-up",
)
ax.legend()
fig.tight_layout()
plt.show()

## 2. Build PyHealth datasets and loaders

We use disjoint groups of 256 training patients and 64 validation patients. Tensor processors preserve numeric features, durations, and event indicators. Reuse fitted training processors for validation.

A batch has features `[B, 4]` and two labels of shape `[B]`. The event field uses a tensor processor, so all-censored datasets are also valid. Trainer weights validation losses by sample count, including a short final batch.


In [ ]:
input_schema = {"features": "tensor"}
output_schema = {"duration": "tensor", "event": "tensor"}
train_dataset = create_sample_dataset(
    samples[:256], input_schema, output_schema, in_memory=True
)
validation_dataset = create_sample_dataset(
    samples[256:],
    input_schema,
    output_schema,
    in_memory=True,
    input_processors=train_dataset.input_processors,
    output_processors=train_dataset.output_processors,
)
train_loader = get_dataloader(train_dataset, batch_size=32, shuffle=True, seed=SEED)
train_eval_loader = get_dataloader(train_dataset, batch_size=32)
validation_loader = get_dataloader(validation_dataset, batch_size=32)
batch = next(iter(validation_loader))
for key in ("features", "duration", "event"):
    print(key, tuple(batch[key].shape))
assert {s["patient_id"] for s in samples[:256]}.isdisjoint(
    s["patient_id"] for s in samples[256:]
)

## 3. Create and train the model

The hidden layers map patient features to three heads: component weights, latent means, and positive scales. Every head is patient-specific. These components are mathematical building blocks, not established clinical patient groups.

Observed events contribute log density at the event time; censored patients contribute log survival beyond follow-up. `mode=None` lets the existing Trainer optimize this custom loss. Validation loss selects the best checkpoint, with patience three. Density can exceed one, so a negative continuous-time loss is not automatically an error.

Rerunning this cell recreates the model and loaders with the same seed.


In [ ]:
torch.manual_seed(SEED)
train_loader = get_dataloader(train_dataset, batch_size=32, shuffle=True, seed=SEED)
config = {
    "hidden_dims": [32, 32],
    "n_components": 3,
    "activation": "prelu",
    "batch_norm": False,
    "min_scale": 1e-4,
}
model = SurvivalMDN(train_dataset, **config)
trainer = Trainer(model=model, device=DEVICE, enable_logging=False)
initial_loss = trainer.evaluate(train_eval_loader)["loss"]
trainer.train(
    train_loader,
    val_dataloader=validation_loader,
    epochs=EPOCHS,
    monitor="loss",
    monitor_criterion="min",
    patience=3,
    load_best_model_at_last=True,
)
training_loss = trainer.evaluate(train_eval_loader)["loss"]
validation_loss = trainer.evaluate(validation_loader)["loss"]
display(
    pd.DataFrame(
        {
            "measurement": [
                "Initial training loss",
                "Restored training loss",
                "Restored validation loss",
            ],
            "loss": [initial_loss, training_loss, validation_loss],
        }
    )
)
assert np.isfinite([initial_loss, training_loss, validation_loss]).all()

## 4. Predict survival curves

`S(t)` is the probability of remaining event-free beyond time `t`. Prediction needs only features and a grid of requested times. The grid samples a continuous-time model; it does not define training bins.

The following curves are for three validation patients. Different curves express different predicted distributions; they do not establish predictive accuracy from three outcomes. Use `eval()` for evaluation behavior and `no_grad()` when gradients are unnecessary.


In [ ]:
patient_ids = batch["patient_id"][:3]
features = batch["features"][:3]
grid = torch.linspace(0.0, 6.0, 121, device=DEVICE)
model.eval()
with torch.no_grad():
    curves = model.predict_survival(grid, features=features)
assert curves.shape == (3, len(grid))
assert ((curves >= 0) & (curves <= 1)).all()
assert (curves[:, 1:] <= curves[:, :-1] + 1e-7).all()
torch.testing.assert_close(curves[:, 0], torch.ones(3, device=DEVICE))
fig, ax = plt.subplots(figsize=(7, 4))
for patient_id, curve in zip(patient_ids, curves.cpu().numpy()):
    ax.plot(grid.cpu().numpy(), curve, label=patient_id)
ax.set(
    xlabel="Time (synthetic units)",
    ylabel="Survival probability S(t)",
    ylim=(0, 1.03),
    title="Predicted survival for three validation patients",
)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Inspect the mixture parameters

The weights sum to one for each patient; scales are positive. Means and scales refer to the **latent Gaussian variable**, not directly to event time. The event time is its softplus transformation. Component numbers are arbitrary; they are not clinical labels.

Training uses stable log-space likelihoods. Mathematical and gradient tests live in `tests/core/test_survival_mdn.py`.


In [ ]:
with torch.no_grad():
    parameters = model(features=features)
weights = parameters["log_weights"].exp().cpu()
means = parameters["means"].cpu()
scales = parameters["scales"].cpu()
rows = [
    {
        "patient": patient_ids[i],
        "component": k + 1,
        "weight": weights[i, k].item(),
        "latent_mean": means[i, k].item(),
        "latent_scale": scales[i, k].item(),
    }
    for i in range(len(patient_ids))
    for k in range(config["n_components"])
]
display(pd.DataFrame(rows).round(4))
torch.testing.assert_close(weights.sum(-1), torch.ones(len(patient_ids)))
assert (scales > 0).all()
fig, ax = plt.subplots(figsize=(7, 3))
bottom = np.zeros(len(patient_ids))
for k in range(config["n_components"]):
    values = weights[:, k].numpy()
    ax.bar(patient_ids, values, bottom=bottom, label=f"Component {k + 1}")
    bottom += values
ax.set(
    ylabel="Mixture weight", ylim=(0, 1.05), title="Patient-specific component weights"
)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
plt.show()

## 6. Save and reconstruct the model

A state dictionary contains learned weights and buffers, not constructor settings or all fitted processor metadata. Save configuration separately and reconstruct with a compatible processed dataset. Here we reuse the training dataset's schema and processors.

Artifacts go under `output/survival_mdn_notebook/` in your checkout. Rerunning this cell replaces this demonstration's files.


In [ ]:
output_dir = repo / "output/survival_mdn_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "model_config": config,
    "input_schema": input_schema,
    "output_schema": output_schema,
    "feature_width": 4,
    "seed": SEED,
    "time_unit": "arbitrary synthetic unit",
}
(output_dir / "config.json").write_text(json.dumps(metadata, indent=2) + "\n")
torch.save(model.state_dict(), output_dir / "state_dict.pt")
saved = json.loads((output_dir / "config.json").read_text())
restored = SurvivalMDN(train_dataset, **saved["model_config"]).to(DEVICE)
restored.load_state_dict(
    torch.load(output_dir / "state_dict.pt", map_location=DEVICE, weights_only=True)
)
restored.eval()
with torch.no_grad():
    restored_curves = restored.predict_survival(grid, features=features)
torch.testing.assert_close(curves, restored_curves, rtol=0, atol=0)
np.savez(
    output_dir / "survival_curves.npz",
    times=grid.cpu().numpy(),
    survival=curves.cpu().numpy(),
    features=features.cpu().numpy(),
)
print("Checkpoint reload: predictions match exactly")
print("Artifacts:", output_dir)

## Review and next steps

- Read `pyhealth/models/survival_mdn.py`: start with `forward` and `predict_survival`, then the distribution helpers.
- Run the offline tests: `python -m unittest discover -t tests -s tests/core -p 'test_survival_mdn.py' -v`.
- Change `EPOCHS`, hidden widths, or component count and rerun from the training cell onward.

Reference: [Han, Goldstein, and Ranganath (2022), Survival Mixture Density Networks](https://proceedings.mlr.press/v182/han22a.html).
